# LeetCode 1560 — Most Visited Sector in a Circular Track bằng Python

**Link đề:** https://leetcode.com/problems/most-visited-sector-in-a-circular-track/
**Độ khó:** Easy · **Chủ đề:** Array, Simulation

```python
class Solution:
    def mostVisited(self, n: int, rounds: list[int]) -> list[int]:
        ...
```

### Mục tiêu học
1. Mô hình hóa **đường tròn** với nhãn `1..n`: bước tiếp theo của `x` là `x % n + 1` (quay vòng từ `n` về `1`).
2. Viết mô phỏng đếm lượt ghé thăm bằng **mảng đếm** (`[0] * (n + 1)`), tìm `max` và lọc các sector đạt max.
3. Tìm ra **insight mạnh**: chỉ `rounds[0]` (điểm xuất phát) và `rounds[-1]` (điểm kết thúc) quyết định đáp án.
4. Từ đó viết lời giải `O(n)` không phụ thuộc số vòng `m`; xử lý hai trường hợp `start <= end` và `start > end`.
5. Trace, chứng minh, Time/Space, test đối chiếu, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — sector nào được đi qua nhiều nhất?

Có một đường chạy **vòng tròn** gồm `n` sector đánh số `1..n`. Cuộc thi gồm `m` vòng (round). Vòng thứ `i` bắt đầu tại sector `rounds[i-1]` và kết thúc tại `rounds[i]`. Người chạy luôn đi theo chiều **tăng dần** số sector (sau `n` là `1`).

Trả về **danh sách các sector được ghé thăm nhiều nhất**, sắp xếp tăng dần.

| Ví dụ | `n` | `rounds` | Output |
|---|---|---|---|
| 1 | 4 | `[1,3,1,2]` | `[1,2]` |
| 2 | 2 | `[2,1,2,1,2,1,2,1,2]` | `[2]` |
| 3 | 7 | `[1,3,5,7]` | `[1,2,3,4,5,6,7]` |

Giải thích ví dụ 1: đi `1 → 2 → 3` (hết vòng 1) `→ 4 → 1` (hết vòng 2) `→ 2` (hết vòng 3). Sector 1 và 2 được ghé 2 lần, sector 3 và 4 chỉ 1 lần.

**Constraints:** `2 <= n <= 100`, `1 <= m <= 100`, `rounds.length == m + 1`, `1 <= rounds[i] <= n`, `rounds[i] != rounds[i+1]`.

Lưu ý cách đếm: sector **xuất phát** `rounds[0]` được tính là đã ghé 1 lần; điểm cuối của vòng này cũng là điểm đầu của vòng sau nhưng **chỉ tính 1 lần** (không đếm đôi).

In [1]:
n, rounds = 4, [1, 3, 1, 2]
print("Số vòng m =", len(rounds) - 1)
for i in range(1, len(rounds)):
    print(f"vòng {i}: từ sector {rounds[i-1]} đến sector {rounds[i]}")

Số vòng m = 3
vòng 1: từ sector 1 đến sector 3
vòng 2: từ sector 3 đến sector 1
vòng 3: từ sector 1 đến sector 2


## 2. Nền tảng Python cần dùng

### 2.1. Đi vòng tròn: `x % n + 1`

Với nhãn `1..n`, sector kế tiếp của `x` là `x + 1`, riêng `n` quay về `1`. Công thức gọn: `x % n + 1`.
- `x < n`: `x % n = x` → `x + 1` ✓.
- `x = n`: `n % n = 0` → `1` ✓.

In [2]:
n = 4
x = 2
path = [x]
for _ in range(6):
    x = x % n + 1
    path.append(x)
print("Đi 6 bước từ 2 trên vòng 4 sector:", path)

Đi 6 bước từ 2 trên vòng 4 sector: [2, 3, 4, 1, 2, 3, 4]


### 2.2. Mảng đếm `[0] * (n + 1)`

Để đếm số lần ghé từng sector `1..n`, tạo list dài `n + 1` và **bỏ trống ô 0** — như vậy `count[s]` ứng đúng với sector `s`, không cần trừ 1.

In [3]:
n = 4
count = [0] * (n + 1)
for s in [1, 2, 3, 4, 1, 2]:
    count[s] += 1
print("count =", count, "(ô 0 bỏ trống)")

count = [0, 2, 2, 1, 1] (ô 0 bỏ trống)


### 2.3. `max` và lọc bằng list comprehension

`max(iterable)` cho giá trị lớn nhất. Lọc các sector đạt max: `[s for s in range(1, n+1) if count[s] == best]` — tự động **tăng dần** vì duyệt `s` tăng dần.

In [4]:
count = [0, 2, 2, 1, 1]
best = max(count[1:])
print("max =", best)
print("Sector đạt max:", [s for s in range(1, len(count)) if count[s] == best])

max = 2
Sector đạt max: [1, 2]


`count[1:]` bỏ ô 0 trước khi tìm max (ô 0 luôn là 0 nên ở đây không ảnh hưởng, nhưng viết vậy cho rõ ý).

### 2.4. `list(range(a, b + 1))` và nối list `+`

`range(a, b + 1)` gồm `a, a+1, ..., b` (phải `+1` vì `range` không gồm điểm cuối). Hai list nối bằng `+`.

In [5]:
print(list(range(2, 5)), list(range(2, 5 + 1)))
print(list(range(1, 3)) + list(range(6, 8)))

[2, 3, 4] [2, 3, 4, 5]
[1, 2, 6, 7]


## 3. Cách 1 — Mô phỏng từng bước (trực giác)

Làm đúng như đề: bắt đầu tại `rounds[0]` (đếm 1 lần), với mỗi vòng đi từng bước `x = x % n + 1` cho đến khi tới `rounds[i]`, mỗi bước đếm sector vừa tới.

In [6]:
def most_visited_simulate(n: int, rounds: list[int]) -> list[int]:
    count = [0] * (n + 1)
    x = rounds[0]
    count[x] += 1                         # điểm xuất phát
    for target in rounds[1:]:
        while x != target:
            x = x % n + 1                 # bước sang sector kế tiếp
            count[x] += 1
    best = max(count)
    return [s for s in range(1, n + 1) if count[s] == best]

print(most_visited_simulate(4, [1, 3, 1, 2]))
print(most_visited_simulate(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]))
print(most_visited_simulate(7, [1, 3, 5, 7]))

[1, 2]
[2]
[1, 2, 3, 4, 5, 6, 7]


### 3.1. Trace mô phỏng ví dụ 1

In [7]:
def trace_simulate(n, rounds):
    count = [0] * (n + 1)
    x = rounds[0]; count[x] += 1
    print(f"Xuất phát tại {x}: count={count[1:]}")
    for r, target in enumerate(rounds[1:], start=1):
        path = []
        while x != target:
            x = x % n + 1; count[x] += 1; path.append(x)
        print(f"Vòng {r}: đi qua {path} → count={count[1:]}")
    best = max(count)
    ans = [s for s in range(1, n + 1) if count[s] == best]
    print("max =", best, "→ đáp án", ans)
    return ans

assert trace_simulate(4, [1, 3, 1, 2]) == [1, 2]

Xuất phát tại 1: count=[1, 0, 0, 0]
Vòng 1: đi qua [2, 3] → count=[1, 1, 1, 0]
Vòng 2: đi qua [4, 1] → count=[2, 1, 1, 1]
Vòng 3: đi qua [2] → count=[2, 2, 1, 1]
max = 2 → đáp án [1, 2]


`count[1:]` in sector `1..n` cho dễ đọc (bỏ ô 0).

**Chi phí:** mỗi vòng đi tối đa `n - 1` bước → tổng `O(m·n)` + tìm max `O(n)`. Với `m, n ≤ 100` chỉ ~10⁴ bước — quá nhanh. Nhưng có một quan sát làm bài này trở nên **đẹp**...

## 4. Insight — các vòng ở giữa không quan trọng!

Nhìn lại mảng `count` sau mô phỏng: hành trình là **một đường đi liên tục** từ `rounds[0]` đến `rounds[-1]`, đi qua một số **vòng tròn trọn vẹn** ở giữa. Mỗi vòng trọn vẹn cộng **đều 1** cho **mọi** sector → không làm thay đổi sector nào nhiều nhất.

Vậy chỉ phần "lẻ" còn lại quyết định: đoạn từ `start = rounds[0]` đi tới `end = rounds[-1]` (theo chiều tăng, có thể vòng qua `n → 1`). Các sector trên đoạn này được ghé **nhiều hơn đúng 1 lần** so với các sector khác.

Kiểm chứng: so sánh `count` của ví dụ 2 — `rounds` dài, nhưng `start = 2`, `end = 2`.

In [8]:
def visit_counts(n, rounds):
    count = [0] * (n + 1); x = rounds[0]; count[x] += 1
    for target in rounds[1:]:
        while x != target:
            x = x % n + 1; count[x] += 1
    return count[1:]

print("Ví dụ 1:", visit_counts(4, [1, 3, 1, 2]), "| start=1, end=2")
print("Ví dụ 2:", visit_counts(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]), "| start=2, end=2")
print("Ví dụ 3:", visit_counts(7, [1, 3, 5, 7]), "| start=1, end=7")
print("Giữa thay đổi, đầu-cuối giữ nguyên:", visit_counts(4, [1, 3, 2, 4, 3, 2]), "vs", visit_counts(4, [1, 2]))

Ví dụ 1: [2, 2, 1, 1] | start=1, end=2
Ví dụ 2: [4, 5] | start=2, end=2
Ví dụ 3: [1, 1, 1, 1, 1, 1, 1] | start=1, end=7
Giữa thay đổi, đầu-cuối giữ nguyên: [4, 4, 3, 3] vs [1, 1, 0, 0]


Dòng cuối: hai hành trình rất khác nhau (`[1,3,2,4,3,2]` đi rất nhiều vòng, `[1,2]` chỉ một bước), nhưng cùng `start = 1` và cùng `end = 2`. Mảng count khác nhau về **giá trị** (`[4,4,3,3]` so với `[1,1,0,0]`), nhưng **chênh lệch giữa các sector giống hệt nhau** và các sector đạt max đều là `{1, 2}`. Đúng như dự đoán.

## 5. Lời giải chính — Chỉ nhìn `start` và `end`

### 5.1. Hai trường hợp
- **`start <= end`**: đoạn đi là `start, start+1, ..., end` → đáp án `range(start, end + 1)`.
- **`start > end`**: đoạn đi vòng qua `n → 1`: `start, ..., n, 1, ..., end`. Sắp tăng dần: `[1..end] + [start..n]`.

(Trường hợp `start == end` thuộc nhánh đầu, đáp án `[start]` — như ví dụ 2.)

### 5.2. Ghép lại

In [9]:
def most_visited_endpoints(n: int, rounds: list[int]) -> list[int]:
    start, end = rounds[0], rounds[-1]
    if start <= end:
        return list(range(start, end + 1))
    return list(range(1, end + 1)) + list(range(start, n + 1))

print(most_visited_endpoints(4, [1, 3, 1, 2]))
print(most_visited_endpoints(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]))
print(most_visited_endpoints(7, [1, 3, 5, 7]))
print(most_visited_endpoints(5, [4, 2]))      # 4 → 5 → 1 → 2

[1, 2]
[2]
[1, 2, 3, 4, 5, 6, 7]
[1, 2, 4, 5]


## 6. TRACE lời giải chính

| Input | start | end | Nhánh | Đáp án |
|---|---|---|---|---|
| `n=4, [1,3,1,2]` | 1 | 2 | `start <= end` | `[1, 2]` |
| `n=2, [2,1,...,2]` | 2 | 2 | `start <= end` | `[2]` |
| `n=7, [1,3,5,7]` | 1 | 7 | `start <= end` | `[1..7]` |
| `n=5, [4,2]` | 4 | 2 | `start > end` | `[1, 2] + [4, 5] = [1, 2, 4, 5]` |

In [10]:
def trace_endpoints(n, rounds):
    start, end = rounds[0], rounds[-1]
    print(f"n={n}, rounds={rounds}")
    print(f"  start = rounds[0] = {start}, end = rounds[-1] = {end}")
    if start <= end:
        ans = list(range(start, end + 1))
        print(f"  start <= end → range({start}, {end + 1}) = {ans}")
    else:
        left, right = list(range(1, end + 1)), list(range(start, n + 1))
        ans = left + right
        print(f"  start > end → [1..{end}] = {left} + [{start}..{n}] = {right} → {ans}")
    print("  (đối chiếu mô phỏng, count =", visit_counts(n, rounds), ")")
    return ans

assert trace_endpoints(4, [1, 3, 1, 2]) == [1, 2]
assert trace_endpoints(5, [4, 2]) == [1, 2, 4, 5]
assert trace_endpoints(6, [3, 6, 2, 5, 1]) == [1, 3, 4, 5, 6]

n=4, rounds=[1, 3, 1, 2]
  start = rounds[0] = 1, end = rounds[-1] = 2
  start <= end → range(1, 3) = [1, 2]
  (đối chiếu mô phỏng, count = [2, 2, 1, 1] )
n=5, rounds=[4, 2]
  start = rounds[0] = 4, end = rounds[-1] = 2
  start > end → [1..2] = [1, 2] + [4..5] = [4, 5] → [1, 2, 4, 5]
  (đối chiếu mô phỏng, count = [1, 1, 0, 1, 1] )
n=6, rounds=[3, 6, 2, 5, 1]
  start = rounds[0] = 3, end = rounds[-1] = 1
  start > end → [1..1] = [1] + [3..6] = [3, 4, 5, 6] → [1, 3, 4, 5, 6]
  (đối chiếu mô phỏng, count = [2, 1, 2, 2, 2, 2] )


Trace cuối (`n=6`, `start=3`, `end=1`): count từ mô phỏng cho thấy sector 2 thấp hơn hẳn — sector duy nhất **không** nằm trên đoạn `3 → 4 → 5 → 6 → 1`.

## 7. Các lời giải khác

### 7.1. Cách 3 — Đếm đoạn bằng mô phỏng nhưng chỉ từ `start` tới `end`

Kết hợp hai ý: chỉ đi từ `start` đến `end` (một đoạn, không quá `n` bước), thu thập các sector, rồi `sorted`. Viết ngắn, không cần tách hai trường hợp.

In [11]:
def most_visited_walk(n: int, rounds: list[int]) -> list[int]:
    x, end = rounds[0], rounds[-1]
    visited = [x]
    while x != end:
        x = x % n + 1
        visited.append(x)
    return sorted(visited)

print(most_visited_walk(4, [1, 3, 1, 2]), most_visited_walk(5, [4, 2]))

[1, 2] [1, 2, 4, 5]


### 7.2. Cách 4 — Một dòng lọc theo điều kiện "thuộc đoạn vòng tròn"

Sector `s` thuộc đoạn đi từ `start` tới `end` (theo chiều tăng, vòng tròn) ⇔ khoảng cách vòng `(s - start) % n` **không vượt quá** `(end - start) % n`.

In [12]:
def most_visited_modular(n: int, rounds: list[int]) -> list[int]:
    start, end = rounds[0], rounds[-1]
    span = (end - start) % n
    return [s for s in range(1, n + 1) if (s - start) % n <= span]

print(most_visited_modular(4, [1, 3, 1, 2]), most_visited_modular(5, [4, 2]), most_visited_modular(7, [1, 3, 5, 7]))
print("Ví dụ (s - start) % n với n=5, start=4:", {s: (s - 4) % 5 for s in range(1, 6)})

[1, 2] [1, 2, 4, 5] [1, 2, 3, 4, 5, 6, 7]
Ví dụ (s - start) % n với n=5, start=4: {1: 2, 2: 3, 3: 4, 4: 0, 5: 1}


`{khóa: giá_trị for ...}` là **dict comprehension**. Phép `%` của Python với số âm vẫn trả kết quả trong `[0, n)` (ví dụ `(1 - 4) % 5 = 2`), rất tiện cho bài vòng tròn.

### 7.3. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng đầy đủ | đếm mọi bước | `O(m·n)` | `O(n)` | đúng gợi ý đề, dễ kiểm chứng |
| 2. **Đầu–cuối** | 2 trường hợp `range` | **`O(n)`** | `O(n)` (kết quả) | **nên dùng** |
| 3. Đi một đoạn | từ start tới end rồi sort | `O(n log n)` | `O(n)` | không cần tách nhánh |
| 4. Modular | `(s-start)%n <= span` | `O(n)` | `O(n)` | gọn, luyện `%` vòng tròn |

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def mostVisited(self, n: int, rounds: list[int]) -> list[int]:
        start, end = rounds[0], rounds[-1]
        if start <= end:
            return list(range(start, end + 1))
        return list(range(1, end + 1)) + list(range(start, n + 1))

sol = Solution()
print(sol.mostVisited(4, [1, 3, 1, 2]))
print(sol.mostVisited(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]))
print(sol.mostVisited(7, [1, 3, 5, 7]))

[1, 2]
[2]
[1, 2, 3, 4, 5, 6, 7]


## 9. Vì sao đúng?

Ghép tất cả các vòng lại, người chạy đi **một đường liên tục** theo chiều tăng từ `start` tới `end`, tổng cộng `L` bước. Viết `L = q·n + r` với `0 <= r < n`, trong đó `r = (end - start) % n`.

- Đường đi gồm sector `start` (lượt đầu), rồi `L` sector kế tiếp. Chia thành `q` **vòng trọn vẹn** (`q·n` bước — mỗi sector được ghé đúng `q` lần) và `r` bước còn lại.
- Tổng hợp lại: các sector trên đoạn `start, start+1, ..., end` (gồm `r + 1` sector, tính theo vòng tròn) được ghé **`q + 1`** lần; mọi sector khác được ghé **`q`** lần.
- Vì `r + 1 >= 1`, tập sector đạt max chính là đoạn từ `start` tới `end` — đúng điều lời giải trả về. Không phụ thuộc `rounds[1..m-1]`.

Ràng buộc `rounds[i] != rounds[i+1]` đảm bảo mỗi vòng đi ít nhất 1 bước, nhưng lập luận trên đúng cả khi không có ràng buộc này.

In [14]:
# Kiểm chứng lập luận: số lần ghé = q hoặc q + 1
n, rounds = 5, [2, 4, 1, 3, 2, 5]
cnt = visit_counts(n, rounds)
print("count =", cnt, "| các giá trị khác nhau:", sorted(set(cnt)))

count = [2, 3, 3, 3, 3] | các giá trị khác nhau: [2, 3]


Đúng như chứng minh: chỉ có **hai** giá trị đếm, chênh nhau 1.

## 10. Time / Space

- **Lời giải chính:** chỉ đọc `rounds[0]`, `rounds[-1]` (`O(1)`), tạo list kết quả tối đa `n` phần tử → **Time `O(n)`**, **Space `O(n)`** cho kết quả (không có bộ nhớ phụ nào khác). Không phụ thuộc `m`!
- **Mô phỏng:** `O(m·n)` time, `O(n)` space cho mảng đếm.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.mostVisited(4, [1, 3, 1, 2]), [1, 2])
check("đề 2", sol.mostVisited(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]), [2])
check("đề 3", sol.mostVisited(7, [1, 3, 5, 7]), [1, 2, 3, 4, 5, 6, 7])
check("1 vòng, không qua n", sol.mostVisited(5, [2, 4]), [2, 3, 4])
check("1 vòng, qua n→1", sol.mostVisited(5, [4, 2]), [1, 2, 4, 5])
check("start == end", sol.mostVisited(3, [3, 1, 3]), [3])
check("end = n", sol.mostVisited(3, [1, 3]), [1, 2, 3])
check("start = n, end = 1", sol.mostVisited(4, [4, 1]), [1, 4])
check("n = 2", sol.mostVisited(2, [1, 2]), [1, 2])

PASS đề 1: [1, 2]
PASS đề 2: [2]
PASS đề 3: [1, 2, 3, 4, 5, 6, 7]
PASS 1 vòng, không qua n: [2, 3, 4]
PASS 1 vòng, qua n→1: [1, 2, 4, 5]
PASS start == end: [3]
PASS end = n: [1, 2, 3]
PASS start = n, end = 1: [1, 4]
PASS n = 2: [1, 2]


**Phản ví dụ cho cách "chỉ đếm điểm kết thúc của mỗi vòng":** một số người mới đếm `rounds[i]` (các mốc) thay vì mọi sector đi qua.

In [16]:
def most_visited_count_marks_wrong(n, rounds):
    count = [0] * (n + 1)
    for r in rounds:
        count[r] += 1
    best = max(count)
    return [s for s in range(1, n + 1) if count[s] == best]

for n, rounds in [(4, [1, 3, 1, 2]), (7, [1, 3, 5, 7]), (5, [4, 2])]:
    print(n, rounds, "| đếm mốc:", most_visited_count_marks_wrong(n, rounds), "| đúng:", sol.mostVisited(n, rounds))

4 [1, 3, 1, 2] | đếm mốc: [1] | đúng: [1, 2]
7 [1, 3, 5, 7] | đếm mốc: [1, 3, 5, 7] | đúng: [1, 2, 3, 4, 5, 6, 7]
5 [4, 2] | đếm mốc: [2, 4] | đúng: [1, 2, 4, 5]


### 11.1. Đối chiếu 4 cách trên input ngẫu nhiên

Sinh `rounds` thỏa ràng buộc `rounds[i] != rounds[i+1]`.

In [17]:
import random
rng = random.Random(1560)
for _ in range(2000):
    n = rng.randint(2, 12)
    m = rng.randint(1, 15)
    rounds = [rng.randint(1, n)]
    for _ in range(m):
        nxt = rng.randint(1, n)
        while nxt == rounds[-1]:
            nxt = rng.randint(1, n)
        rounds.append(nxt)
    exp = most_visited_simulate(n, rounds)
    assert most_visited_endpoints(n, rounds) == exp
    assert most_visited_walk(n, rounds) == exp
    assert most_visited_modular(n, rounds) == exp
    assert Solution().mostVisited(n, rounds) == exp
print("2000 test ngẫu nhiên: 4 cách khớp với mô phỏng")

2000 test ngẫu nhiên: 4 cách khớp với mô phỏng


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Đếm đôi điểm nối giữa hai vòng
Mô phỏng mỗi vòng **từ `rounds[i-1]` đến `rounds[i]` bao gồm cả hai đầu** → mốc giữa bị đếm 2 lần.

In [18]:
def simulate_double_count_bug(n, rounds):
    count = [0] * (n + 1)
    for i in range(1, len(rounds)):
        x = rounds[i - 1]; count[x] += 1           # SAI: đếm lại điểm đầu mỗi vòng
        while x != rounds[i]:
            x = x % n + 1; count[x] += 1
    best = max(count)
    return [s for s in range(1, n + 1) if count[s] == best]

print("lỗi:", simulate_double_count_bug(4, [1, 3, 1, 2]), "| đúng:", most_visited_simulate(4, [1, 3, 1, 2]))

lỗi: [1] | đúng: [1, 2]


**Sửa:** chỉ đếm `rounds[0]` một lần ở đầu; trong mỗi vòng chỉ đếm sector **sau khi bước**.

### Lỗi 2 — Công thức bước sai: `(x + 1) % n`
Với nhãn `1..n`, `(x + 1) % n` cho `n - 1 → 0` (sector 0 không tồn tại!) và không bao giờ ra `n`. Đúng là `x % n + 1`.

In [19]:
n = 4
print("(x + 1) % n :", [(x + 1) % n for x in range(1, n + 1)])
print("x % n + 1   :", [x % n + 1 for x in range(1, n + 1)])

(x + 1) % n : [2, 3, 0, 1]
x % n + 1   : [2, 3, 4, 1]


### Lỗi 3 — Nhánh `start > end` trả về theo thứ tự đi chứ không sắp tăng
`list(range(start, n + 1)) + list(range(1, end + 1))` → `[4, 5, 1, 2]` — đề yêu cầu **tăng dần**. **Sửa:** đặt `[1..end]` trước.

### Lỗi 4 — Quên `+ 1` ở `range`
`range(start, end)` bỏ mất `end`; `range(start, n)` bỏ mất sector `n`.

In [20]:
def endpoints_off_by_one_bug(n, rounds):
    start, end = rounds[0], rounds[-1]
    if start <= end:
        return list(range(start, end))                         # SAI
    return list(range(1, end)) + list(range(start, n))          # SAI

print("lỗi:", endpoints_off_by_one_bug(5, [4, 2]), "| đúng:", most_visited_endpoints(5, [4, 2]))
print("lỗi:", endpoints_off_by_one_bug(4, [1, 3, 1, 2]), "| đúng:", most_visited_endpoints(4, [1, 3, 1, 2]))

lỗi: [1, 4] | đúng: [1, 2, 4, 5]
lỗi: [1] | đúng: [1, 2]


## 13. Mẹo nhận dạng pattern

- **Vòng tròn nhãn `1..n`**: bước tiếp `x % n + 1`; khoảng cách theo chiều tăng `(b - a) % n`.
- Mô phỏng có nhiều "vòng trọn vẹn" cộng đều cho mọi phần tử → **bỏ qua phần trọn vẹn**, chỉ xét phần dư. (Cùng ý tưởng với `k % n` khi xoay mảng `k` lần — LeetCode 189.)
- Hỏi "phần tử nào **nhiều nhất**" (không hỏi số lần cụ thể) → các phép cộng đều không ảnh hưởng → tìm cái làm **lệch**.
- Đoạn trên vòng tròn `[a..b]` khi `a > b` → tách thành `[1..b] ∪ [a..n]`.
- Constraints nhỏ → mô phỏng luôn là **oracle** tốt để kiểm chứng insight.

## 14. Biến thể

### 14.1. Trả về cả số lần ghé tối đa
Từ chứng minh: `L = (tổng số bước)`, `q = L // n`, max = `q + 1`. Tổng số bước = tổng `(rounds[i] - rounds[i-1]) % n`.

In [21]:
def max_visits(n, rounds):
    steps = sum((rounds[i] - rounds[i - 1]) % n for i in range(1, len(rounds)))
    return steps // n + 1

for n, rounds in [(4, [1, 3, 1, 2]), (2, [2, 1, 2, 1, 2, 1, 2, 1, 2]), (7, [1, 3, 5, 7])]:
    print(n, rounds, "→ max =", max_visits(n, rounds), "| mô phỏng:", max(visit_counts(n, rounds)))

4 [1, 3, 1, 2] → max = 2 | mô phỏng: 2
2 [2, 1, 2, 1, 2, 1, 2, 1, 2] → max = 5 | mô phỏng: 5
7 [1, 3, 5, 7] → max = 1 | mô phỏng: 1


### 14.2. Sector **ít** được ghé nhất
Là phần bù của đoạn `start..end`. Nếu đoạn phủ hết vòng (`(end - start) % n == n - 1`) thì mọi sector bằng nhau.

In [22]:
def least_visited(n, rounds):
    most = set(most_visited_endpoints(n, rounds))
    rest = [s for s in range(1, n + 1) if s not in most]
    return rest if rest else list(range(1, n + 1))

print(least_visited(4, [1, 3, 1, 2]), least_visited(7, [1, 3, 5, 7]))

[3, 4] [1, 2, 3, 4, 5, 6, 7]


### 14.3. Đi theo chiều **giảm** (thuận chiều kim đồng hồ)
Bước tiếp `(x - 2) % n + 1`; đoạn từ `start` **giảm** tới `end`. Xem bài tập 6.

## 15. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
`n = 6`, `rounds = [5, 2, 4]`. Liệt kê đường đi, đếm số lần ghé mỗi sector, cho đáp án. Kiểm tra lại bằng quy tắc start/end.

### Bài 2 — Đoán output
```python
n = 5
print([x % n + 1 for x in [1, 4, 5]], (2 - 4) % 5, (4 - 2) % 5)
```

### Bài 3 — TODO: viết lời giải start/end

In [23]:
def exercise_most_visited(n: int, rounds: list[int]) -> list[int]:
    # TODO 1: start, end = rounds[0], rounds[-1]
    # TODO 2: start <= end → range(start, end + 1)
    # TODO 3: ngược lại → [1..end] + [start..n]
    pass

# assert exercise_most_visited(4, [1, 3, 1, 2]) == [1, 2]
# assert exercise_most_visited(5, [4, 2]) == [1, 2, 4, 5]

### Bài 4 — TODO: mô phỏng trả về mảng đếm
Trả về list `count` độ dài `n` (sector `1..n`) — dùng làm oracle.

In [24]:
def exercise_visit_counts(n: int, rounds: list[int]) -> list[int]:
    # TODO: đếm rounds[0] một lần, mỗi bước x = x % n + 1 thì đếm x
    pass

# assert exercise_visit_counts(4, [1, 3, 1, 2]) == [2, 2, 1, 1]

### Bài 5 — TODO: số lần ghé tối đa không mô phỏng
Trả về **số lần** ghé của sector được ghé nhiều nhất, `O(m)`.

In [25]:
def exercise_max_visits(n: int, rounds: list[int]) -> int:
    # TODO: tổng số bước = sum((rounds[i] - rounds[i-1]) % n); max = steps // n + 1
    pass

# assert exercise_max_visits(4, [1, 3, 1, 2]) == 2
# assert exercise_max_visits(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]) == 5

### Bài 6 — Mini challenge: chạy theo chiều giảm
Cùng đề, nhưng người chạy đi theo chiều **giảm** (`3 → 2 → 1 → n → n-1 ...`). Trả các sector được ghé nhiều nhất, tăng dần.

In [26]:
def exercise_most_visited_clockwise(n: int, rounds: list[int]) -> list[int]:
    # TODO: chỉ start/end quan trọng; đoạn đi giảm từ start tới end
    pass

# assert exercise_most_visited_clockwise(4, [1, 3]) == [1, 3, 4]      # 1 → 4 → 3
# assert exercise_most_visited_clockwise(5, [4, 2]) == [2, 3, 4]

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đi `5 → 6 → 1 → 2`, rồi `2 → 3 → 4`. (2) Đếm từng sector. (3) `start = 5 > end = 4`.

**Bài 2:** (1) `x % n + 1` quay vòng. (2) `%` của số âm trong Python cho kết quả không âm. (3) Tính từng số.

**Bài 3:** (1) Chỉ cần hai đầu. (2) Hai nhánh. (3) Xem mục 5.

**Bài 4:** (1) Như cách 1. (2) Trả `count[1:]`. (3) Không đếm đôi mốc giữa.

**Bài 5:** (1) Mục 9: max = `q + 1`. (2) Bước của vòng `i` là `(rounds[i] - rounds[i-1]) % n`. (3) Cộng lại rồi `// n + 1`.

**Bài 6:** (1) Đổi vai trò: đi giảm từ `start` tới `end` ≡ đi tăng từ `end` tới `start`. (2) Áp dụng lời giải chính với `(end, start)` đổi chỗ. (3) `exercise_most_visited(n, [end, start])`.

## 17. Đáp án tham khảo

**Bài 1:** Xuất phát tại 5 (đếm 1 lần); vòng 1 đi `6, 1, 2`; vòng 2 đi `3, 4`. Đường đi đầy đủ `5, 6, 1, 2, 3, 4` — mỗi sector được ghé đúng 1 lần, count = `[1,1,1,1,1,1]` → đáp án `[1, 2, 3, 4, 5, 6]`. Quy tắc: `start = 5 > end = 4` → `[1..4] + [5..6]` = `[1, 2, 3, 4, 5, 6]` ✓ (đoạn từ 5 tới 4 phủ trọn vòng).

**Bài 2:** `[2, 5, 1] 3 2`.

In [27]:
def answer_most_visited(n, rounds):
    start, end = rounds[0], rounds[-1]
    if start <= end:
        return list(range(start, end + 1))
    return list(range(1, end + 1)) + list(range(start, n + 1))

def answer_visit_counts(n, rounds):
    count = [0] * (n + 1); x = rounds[0]; count[x] += 1
    for target in rounds[1:]:
        while x != target:
            x = x % n + 1; count[x] += 1
    return count[1:]

def answer_max_visits(n, rounds):
    steps = sum((rounds[i] - rounds[i - 1]) % n for i in range(1, len(rounds)))
    return steps // n + 1

def answer_most_visited_clockwise(n, rounds):
    start, end = rounds[0], rounds[-1]
    return answer_most_visited(n, [end, start])

def simulate_clockwise(n, rounds):              # oracle cho bài 6
    count = [0] * (n + 1); x = rounds[0]; count[x] += 1
    for target in rounds[1:]:
        while x != target:
            x = (x - 2) % n + 1; count[x] += 1
    best = max(count)
    return [s for s in range(1, n + 1) if count[s] == best]

print("Bài 2:", [x % 5 + 1 for x in [1, 4, 5]], (2 - 4) % 5, (4 - 2) % 5)
assert answer_most_visited(6, [5, 2, 4]) == [1, 2, 3, 4, 5, 6]
assert answer_visit_counts(6, [5, 2, 4]) == [1, 1, 1, 1, 1, 1]
assert answer_most_visited(4, [1, 3, 1, 2]) == [1, 2]
assert answer_most_visited(5, [4, 2]) == [1, 2, 4, 5]
assert answer_visit_counts(4, [1, 3, 1, 2]) == [2, 2, 1, 1]
assert answer_max_visits(4, [1, 3, 1, 2]) == 2
assert answer_max_visits(2, [2, 1, 2, 1, 2, 1, 2, 1, 2]) == 5
assert answer_most_visited_clockwise(4, [1, 3]) == [1, 3, 4]
assert answer_most_visited_clockwise(5, [4, 2]) == [2, 3, 4]
rng = random.Random(8)
for _ in range(1000):
    n = rng.randint(2, 9); rounds = [rng.randint(1, n)]
    for _ in range(rng.randint(1, 8)):
        nxt = rng.randint(1, n)
        while nxt == rounds[-1]:
            nxt = rng.randint(1, n)
        rounds.append(nxt)
    assert answer_max_visits(n, rounds) == max(answer_visit_counts(n, rounds))
    assert answer_most_visited_clockwise(n, rounds) == simulate_clockwise(n, rounds)
print("Tất cả đáp án PASS")

Bài 2: [2, 5, 1] 3 2
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def mostVisited(self, n: int, rounds: list[int]) -> list[int]:
        start, end = rounds[0], rounds[-1]          # chỉ hai đầu quan trọng
        if start <= end:
            return list(range(start, end + 1))
        return list(range(1, end + 1)) + list(range(start, n + 1))
```

**Câu thần chú:** *"Vòng trọn vẹn cộng đều cho mọi người — chỉ đoạn lẻ từ start tới end là được thêm 1."*

**Checklist trước khi submit**
- [ ] Dùng `rounds[0]` và `rounds[-1]`, không phải `rounds[1]`.
- [ ] Nhánh `start > end`: `[1..end]` đứng **trước** `[start..n]` (tăng dần).
- [ ] `range(..., end + 1)` và `range(..., n + 1)` — đừng quên `+ 1`.
- [ ] Test `start == end`, `start = n`, đoạn phủ trọn vòng.
- [ ] Nếu mô phỏng: bước `x % n + 1`, không đếm đôi mốc giữa các vòng.
- [ ] Đúng tên method `mostVisited`.